In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "data" / "final" / "conflict_country_year_base.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
df.head()

Dataset loaded successfully.
Shape: (6737, 22)


,country_id,country,year,region,main_government_name,state_based_conflict_exists,state_based_dyad_count,state_based_deaths_best,intrastate_conflict_exists,intrastate_deaths_best,...,non_state_conflict_exists,non_state_dyad_count,non_state_deaths_best,one_sided_violence_exists,one_sided_dyad_count,one_sided_deaths_best,cumulative_organized_violence_deaths_best,ucdp_version,organized_violence_exists,target_conflict_next_year
0,2,United States of America,1989,Americas,Government of United States of America,0,0,0,0,0,...,0,0,0,0,0,0,0,25.1,0,0
1,2,United States of America,1990,Americas,Government of United States of America,0,0,0,0,0,...,0,0,0,0,0,0,0,25.1,0,0
2,2,United States of America,1991,Americas,Government of United States of America,0,0,0,0,0,...,0,0,0,0,0,0,0,25.1,0,0
3,2,United States of America,1992,Americas,Government of United States of America,0,0,0,0,0,...,0,0,0,0,0,0,0,25.1,0,0
4,2,United States of America,1993,Americas,Government of United States of America,0,0,0,0,0,...,0,0,0,0,0,0,0,25.1,0,0


In [2]:
TARGET_COLUMN = "target_conflict_next_year"

FEATURE_COLUMNS = [
    "year",
    "state_based_conflict_exists",
    "state_based_dyad_count",
    "state_based_deaths_best",
    "intrastate_conflict_exists",
    "intrastate_deaths_best",
    "interstate_conflict_exists",
    "interstate_deaths_best",
    "non_state_conflict_exists",
    "non_state_dyad_count",
    "non_state_deaths_best",
    "one_sided_violence_exists",
    "one_sided_dyad_count",
    "one_sided_deaths_best",
    "cumulative_organized_violence_deaths_best",
    "organized_violence_exists",
]

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print("Features:", X.shape)
print("Target:", y.shape)
print("\nTarget distribution:")
print(y.value_counts(normalize=True).sort_index().round(4))

Features: (6737, 16)
Target: (6737,)

Target distribution:
target_conflict_next_year
0    0.7051
1    0.2949
Name: proportion, dtype: float64


In [3]:
TRAIN_END_YEAR = 2016

train_mask = df["year"] <= TRAIN_END_YEAR
test_mask = df["year"] > TRAIN_END_YEAR

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

print("Train period:", df.loc[train_mask, "year"].min(), "-", df.loc[train_mask, "year"].max())
print("Test period:", df.loc[test_mask, "year"].min(), "-", df.loc[test_mask, "year"].max())

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).sort_index().round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).sort_index().round(4))

Train period: 1989 - 2016
Test period: 2017 - 2023

Train shape: (5365, 16)
Test shape: (1372, 16)

Train target distribution:
target_conflict_next_year
0    0.7118
1    0.2882
Name: proportion, dtype: float64

Test target distribution:
target_conflict_next_year
0    0.6786
1    0.3214
Name: proportion, dtype: float64


In [6]:
def evaluate_model(model_name, y_true, y_pred):
    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }

    return metrics


results = []

In [7]:
y_pred_persistence_test = df.loc[test_mask, "organized_violence_exists"]

persistence_metrics = evaluate_model(
    "Persistence baseline",
    y_test,
    y_pred_persistence_test
)

results.append(persistence_metrics)

pd.DataFrame(results).round(4)

,model,accuracy,precision,recall,f1_score
0,Persistence baseline,0.9082,0.8571,0.8571,0.8571


In [8]:
logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(X_train, y_train)

y_pred_logistic = logistic_model.predict(X_test)

logistic_metrics = evaluate_model(
    "Logistic Regression",
    y_test,
    y_pred_logistic
)

results.append(logistic_metrics)

pd.DataFrame(results).round(4)

C:\Users\enzo.going\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,model,accuracy,precision,recall,f1_score
0,Persistence baseline,0.9082,0.8571,0.8571,0.8571
1,Logistic Regression,0.9082,0.8571,0.8571,0.8571


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


logistic_scaled_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_scaled_model.fit(X_train, y_train)

y_pred_logistic_scaled = logistic_scaled_model.predict(X_test)

logistic_scaled_metrics = evaluate_model(
    "Logistic Regression scaled",
    y_test,
    y_pred_logistic_scaled
)

results.append(logistic_scaled_metrics)

pd.DataFrame(results).round(4)

,model,accuracy,precision,recall,f1_score
0,Persistence baseline,0.9082,0.8571,0.8571,0.8571
1,Logistic Regression,0.9082,0.8571,0.8571,0.8571
2,Logistic Regression scaled,0.9082,0.8604,0.8526,0.8565


In [10]:
tree_model = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)

tree_metrics = evaluate_model(
    "Decision Tree",
    y_test,
    y_pred_tree
)

results.append(tree_metrics)

pd.DataFrame(results).round(4)

,model,accuracy,precision,recall,f1_score
0,Persistence baseline,0.9082,0.8571,0.8571,0.8571
1,Logistic Regression,0.9082,0.8571,0.8571,0.8571
2,Logistic Regression scaled,0.9082,0.8604,0.8526,0.8565
3,Decision Tree,0.9016,0.9527,0.7302,0.8267


In [11]:
forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

forest_model.fit(X_train, y_train)

y_pred_forest = forest_model.predict(X_test)

forest_metrics = evaluate_model(
    "Random Forest",
    y_test,
    y_pred_forest
)

results.append(forest_metrics)

pd.DataFrame(results).round(4)

,model,accuracy,precision,recall,f1_score
0,Persistence baseline,0.9082,0.8571,0.8571,0.8571
1,Logistic Regression,0.9082,0.8571,0.8571,0.8571
2,Logistic Regression scaled,0.9082,0.8604,0.8526,0.8565
3,Decision Tree,0.9016,0.9527,0.7302,0.8267
4,Random Forest,0.9082,0.8571,0.8571,0.8571
